In [1]:
!pip install sacrebleu gdown -q

import os
import re
import math
import time
import random
import unicodedata
from collections import Counter

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
import sacrebleu

# Reproducibility
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Using CUDA for fast parallel computation
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 6.7 MB/s eta 0:00:00
Using device: cpu


# 1. Data Loading & Preprocessing


In [3]:
import os
import glob
import re
import unicodedata
from collections import Counter
import pandas as pd

# 1. Download the public Google Drive folder into './dataset'
FOLDER_URL = "https://drive.google.com/drive/folders/1R4pOtvDEYyXtjLgVsT2RsmaORKOzoEq2?usp=drive_link"
!gdown --folder "{FOLDER_URL}" -O ./dataset

# Special tokens & fixed indices- (padding, start of statement, end of statement, unknown)
PAD_TOKEN, SOS_TOKEN, EOS_TOKEN, UNK_TOKEN = "<pad>", "<sos>", "<eos>", "<unk>"
PAD_IDX, SOS_IDX, EOS_IDX, UNK_IDX = 0, 1, 2, 3

# NFD is Normalization Form Canonical Decomposition. It splits é to (e + ')
# Mn is Mark, Nonspacing
# We check if each character is Mn. We discard the Mn and keep the non-Mn ones.
def unicode_to_ascii(s):
    """Normalize unicode characters and strip accents."""
    return ''.join(c for c in unicodedata.normalize('NFD', str(s)) if unicodedata.category(c) != 'Mn')

def normalize_text(s):
    """Lowercase, detach punctuation, and remove non-alphabetic characters."""
    s = unicode_to_ascii(s.lower().strip())
    s = re.sub(r"([.!?])", r" \1", s)       # Eg: Turns "Hello!" to "Hello !". This helps the model to treat "!" as a separate token
    s = re.sub(r"[^a-zA-Z.!?]+", r" ", s)       # '^' means NOT & '+' means it matches groups of these characters all at once. This statement removes all the irrelevant non-alphabetic characters.
    return s.strip()

# Download the dataset from the public Google Drive link
def find_file(filename):
    matches = glob.glob(f"./dataset/**/{filename}", recursive=True)
    if not matches:
        raise FileNotFoundError(f"Could not find {filename} in ./dataset")
    return matches[0]

train_path = find_file("en_fr_train.csv")
val_path   = find_file("en_fr_val.csv")
test_path  = find_file("en_fr_test.csv")

df_train = pd.read_csv(train_path)
df_val   = pd.read_csv(val_path)
df_test  = pd.read_csv(test_path)

# Source(src) is the input data/sequence sent into the encoder
# Target(tgt) is the input data/sequence sent into the decoder
def extract_pairs(df):
    # Drop rows where either column 0 or column 1 has missing data
    df_clean = df.dropna(subset=[df.columns[0], df.columns[1]])

    src = [normalize_text(s) for s in df_clean.iloc[:, 0].astype(str)]
    tgt = [normalize_text(t) for t in df_clean.iloc[:, 1].astype(str)]

    return src, tgt

train_src, train_tgt = extract_pairs(df_train)
val_src, val_tgt     = extract_pairs(df_val)
test_src, test_tgt   = extract_pairs(df_test)

print(f"\nDataset sizes:")
print(f" - Train pairs: {len(train_src)}")
print(f" - Val pairs:   {len(val_src)}")
print(f" - Test pairs:  {len(test_src)}")

# 3. Vocabulary Construction (built strictly from training set)
class Vocab:
    def __init__(self, min_freq=2):
        self.min_freq = min_freq
        self.itos = {PAD_IDX: PAD_TOKEN, SOS_IDX: SOS_TOKEN, EOS_IDX: EOS_TOKEN, UNK_IDX: UNK_TOKEN}
        self.stoi = {v: k for k, v in self.itos.items()}

    def build_vocab(self, sentence_list):
        counter = Counter()
        for sent in sentence_list:
            counter.update(sent.split())            # 'counter' isn't a regular dictionary. It internally loops through the list. Thus, it accepts a list/iterable as key to the dictionary.
        for word, count in counter.items():
            if count >= self.min_freq and word not in self.stoi:
                idx = len(self.itos)
                self.itos[idx] = word           # Appends the word to the dictionary itos and stoi, if its freq>min_freq
                self.stoi[word] = idx

    def __len__(self):
        return len(self.itos)

    def numericalize(self, sent):
        return [self.stoi.get(w, UNK_IDX) for w in sent.split()]      # Gets the index (from 'stoi' dictionary) of each word in sentence 'sent'

src_vocab = Vocab(min_freq=2)
tgt_vocab = Vocab(min_freq=2)

src_vocab.build_vocab(train_src)
tgt_vocab.build_vocab(train_tgt))

print(f"\nVocabulary sizes (min_freq=2):")
print(f" - Source (EN) Vocab: {len(src_vocab)}")
print(f" - Target (FR) Vocab: {len(tgt_vocab)}")

Retrieving folder contents
Processing file 1knlGeiSoeQTO02Lmo0WsswX-lF3WYCiz en_fr_test.csv
Processing file 1tMl0dldQfQ8y50B-gGX4myQ_tDneIaTi en_fr_train.csv
Processing file 16y8lV5DrFgrFfpOS6wAzAtGFZwEeg4LF en_fr_val.csv
Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1knlGeiSoeQTO02Lmo0WsswX-lF3WYCiz
To: /content/dataset/en_fr_test.csv
100% 1.74M/1.74M [00:00<00:00, 54.8MB/s]
Downloading...
From: https://drive.google.com/uc?id=1tMl0dldQfQ8y50B-gGX4myQ_tDneIaTi
To: /content/dataset/en_fr_train.csv
100% 48.5M/48.5M [00:00<00:00, 50.2MB/s]
Downloading...
From: https://drive.google.com/uc?id=16y8lV5DrFgrFfpOS6wAzAtGFZwEeg4LF
To: /content/dataset/en_fr_val.csv
100% 205k/205k [00:00<00:00, 4.00MB/s]
Download completed
Loading files:
 - Train: ./dataset/en_fr_train.csv
 - Val:   ./dataset/en_fr_val.csv
 - Test:  ./dataset/en_fr_test.csv

Dataset sizes:
 - Train pairs: 232825
 - Val p

# 3. Dataset and Collate Functions

Getting the tensors for each sentence, containing the indices of the words in the sentence.

Each sentence is made up of different no. of tokens Thus, performing the padding on each sentence. For the tensor to be 2D, we need all sentences to have equal no. of tokens. Thus, we pad the sentences with lesser no. of tokens.


In [4]:
class TranslationDataset(Dataset):
    def __init__(self, src_sents, tgt_sents, src_vocab, tgt_vocab):
        self.pairs = []                         # Creating a list of tuples
        for s, t in zip(src_sents, tgt_sents):
            s_ids = [SOS_IDX] + src_vocab.numericalize(s) + [EOS_IDX]
            t_ids = [SOS_IDX] + tgt_vocab.numericalize(t) + [EOS_IDX]
            self.pairs.append((torch.tensor(s_ids), torch.tensor(t_ids)))     # Appends a pair of (source,target) tensor in the list.
            # A 1D tensor/list containing the indices of the words (in the dictionary stoi) present in the sentence.

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        return self.pairs[idx]

def collate_fn(batch):
    src_batch, tgt_batch = zip(*batch)            # batch is a list of tuples containing the tensors. If batch_size =32, batch is a list of 32 elements.
                                                  # src_batch is a tuple containing the 32 source tensors

    src_padded = pad_sequence(src_batch, batch_first=True, padding_value=PAD_IDX)
    tgt_padded = pad_sequence(tgt_batch, batch_first=True, padding_value=PAD_IDX)
    return src_padded, tgt_padded         # Each is a 2D PyTorch tensor of shape [batch_size, max_src_len]

train_dataset = TranslationDataset(train_src, train_tgt, src_vocab, tgt_vocab)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=collate_fn)

# 4. Encoder, Decoder, Seq2Seq

Due to the absence of Attention in this model, the encoder compresses the entire sentence into the final hidden state. This ruins long-sentence tranlations regardless if we choose LSTM or GRU.


For approximately 200K training samples and 42K vocabulary a GRU would allow faster training(due to its reduced cell parameter). Thus, LSTM would give minimal practical return.

Thus, we choose GRU in this dataset.


In [ ]:
class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hidden_dim, num_layers=1, dropout=0.1):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, emb_dim, padding_idx=PAD_IDX)
        # nn.GRU instead of nn.LSTM
        self.gru = nn.GRU(emb_dim, hidden_dim, num_layers=num_layers, batch_first=True, dropout=dropout if num_layers > 1 else 0.0)
        self.dropout = nn.Dropout(dropout)

    def forward(self, src):
        embedded = self.dropout(self.embedding(src))
        outputs, hidden = self.gru(embedded)
        # Returns only 'hidden' (no cell state 'c' in GRU)
        return hidden

class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hidden_dim, num_layers=1, dropout=0.1):
        super().__init__()
        self.output_dim = output_dim
        self.embedding = nn.Embedding(output_dim, emb_dim, padding_idx=PAD_IDX)
        # nn.GRU instead of nn.LSTM
        self.gru = nn.GRU(emb_dim, hidden_dim, num_layers=num_layers, batch_first=True, dropout=dropout if num_layers > 1 else 0.0)
        self.fc_out = nn.Linear(hidden_dim, output_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, hidden):
        embedded = self.dropout(self.embedding(x))
        output, hidden = self.gru(embedded, hidden)
        prediction = self.fc_out(output.squeeze(1))
        return prediction, hidden

class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device

    def forward(self, src, tgt, teacher_forcing_ratio=0.5):
        # src: [batch_size, src_len]
        # tgt: [batch_size, tgt_len]
        batch_size = src.shape[0]
        tgt_len = tgt.shape[1]
        tgt_vocab_size = self.decoder.output_dim

        # Tensor to store token prediction logits at each decoding step
        outputs = torch.zeros(batch_size, tgt_len, tgt_vocab_size).to(self.device)

        # Encode: extracts the compressed context vector 'hidden'
        hidden = self.encoder(src)

        # First input fed into the decoder is the <sos> token at index 0
        input_tok = tgt[:, 0].unsqueeze(1)  # [batch_size, 1]

        # Autoregressive decoding loop
        for t in range(1, tgt_len):
            prediction, hidden = self.decoder(input_tok, hidden)
            outputs[:, t, :] = prediction

            # Teacher forcing decision
            teacher_force = random.random() < teacher_forcing_ratio
            top1 = prediction.argmax(1).unsqueeze(1)
            input_tok = tgt[:, t].unsqueeze(1) if teacher_force else top1

        return outputs

# 5. Model Training Loop

In [ ]:
EMB_DIM = 64
HIDDEN_DIM = 128
NUM_LAYERS = 2
DROPOUT = 0.2
EPOCHS = 20
LEARNING_RATE = 0.003

enc = Encoder(len(src_vocab), EMB_DIM, HIDDEN_DIM, NUM_LAYERS, DROPOUT)
dec = Decoder(len(tgt_vocab), EMB_DIM, HIDDEN_DIM, NUM_LAYERS, DROPOUT)
model = Seq2Seq(enc, dec, device).to(device)

criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

def train(model, loader, optimizer, criterion, clip=1.0):
    model.train()
    epoch_loss = 0
    for src, tgt in loader:
        src, tgt = src.to(device), tgt.to(device)
        optimizer.zero_grad()

        output = model(src, tgt, teacher_forcing_ratio=0.5)

        # Flatten outputs and target ignoring <sos>
        output_dim = output.shape[-1]
        output = output[:, 1:].reshape(-1, output_dim)
        tgt = tgt[:, 1:].reshape(-1)

        loss = criterion(output, tgt)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), clip)
        optimizer.step()
        epoch_loss += loss.item()

    return epoch_loss / len(loader)

for epoch in range(1, EPOCHS + 1):
    loss = train(model, train_loader, optimizer, criterion)
    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch: {epoch:02d} | Train Loss: {loss:.4f}")

# 6. Inference & BLEU Evaluation

-In standard Seq2Seq without attention, the entire variable-length source
     sequence is squeezed into a single & fixed-size hidden/cell context vector.
   
   For long sentences (length > 15-20), the context vector loses early word
     details, leading to missed clauses, repeated tokens, or early EOS generation.


This motivate us to introduce Bahdnau/Luong Attention in to fix the issues in this translation task.